In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. **Local MCP — SQLite Database: `MCPServerStdio`**
      * 02-05-06. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# Using Model Context Protocol (MCP) to Expose a Local SQLite Database
* Local MCP server providing access to SQLite database 

---

## Key Concepts
* **MCP (Model Context Protocol)** 
    * Open standard created by Anthropic for connecting AI models to external tools and data sources
    * MCP server exposes tools to agent clients
* **`FastMCP`** 
    * High-level `agents.mcp` class for building MCP servers
    * Decorate plain functions with `@mcp_server.tool()` and they become agent-callable tools
* **`MCPServerStdio`** 
    * Launches local MCP server as a subprocess and **communicates via stdin/stdout**
* **`async with` lifecycle** 
    * Server process starts on entry, stops on exit
    * Agent must be created and used inside this block
* **`mcp_servers=`** 
    * Parameter on `Agent`
    * Tools from every connected MCP server and `tools=` entries are all just tools to the model
* **`cache_tools_list=True`**
    * Fetches the tool list once and reuses it across all turns
---

## Imports

In [ ]:
import os
import sqlite3
from pathlib import Path
from agents import Agent, Runner, WebSearchTool, trace
from agents.mcp import MCPServerStdio
from source.agent_loop import run_conversation

---

## Create the Database
* `books.sql` contains a three-table relational schema 

| Table | Columns | Notes |
|---|---|---|
| `authors` | `id`, `first`, `last` | 5 Deitel authors |
| `titles` | `isbn`, `title`, `edition`, `copyright` | 12 books |
| `author_ISBN` | `id`, `isbn` | many-to-many join table |

* Re-running the following cell recreates the database from scratch

In [ ]:
import sqlite3
from pathlib import Path

DB_PATH  = Path('resources') / 'outputs' / 'books.db'
SQL_PATH = Path('resources') / 'books.sql'

with sqlite3.connect(DB_PATH) as connection:
    connection.executescript(SQL_PATH.read_text())

print(f'Database ready: {DB_PATH.relative_to('.')} ({DB_PATH.stat().st_size:,} bytes)')

## Verify Schema and Data

In [ ]:
import sqlite3
from pathlib import Path

with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row

    print('=== authors ===')
    for row in conn.execute('SELECT * FROM authors ORDER BY id'):
        print(f'  {row["id"]}  {row["first"]} {row["last"]}')

    print()
    print('=== titles (newest first) ===')
    for row in conn.execute('SELECT title, edition, copyright FROM titles ORDER BY copyright DESC'):
        print(f'  {row["title"]} ({row["edition"]}e, {row["copyright"]})')

## Local MCP Server
* Write `sqlite_mcp_server.py` to disk using the `%%writefile` Jupyter magic
* `MCPServerStdio` will launch it as a subprocess each time the agent runs
* Server uses `FastMCP` like the client 
    * Guarantees protocol version match

In [ ]:
%%writefile sqlite_mcp_server.py
"""sqlite_mcp_server.py — MCP server exposing a SQLite database as agent tools.

Launched automatically by MCPServerStdio — no need to run this manually.
Accepts an optional database path as a command-line argument. 
Default location: ./resources/outputs/books.db
"""

import json
import sqlite3
import sys
from pathlib import Path
from typing import Annotated

from mcp.server.fastmcp import FastMCP
from pydantic import Field

DB_PATH = Path(sys.argv[1]) if len(sys.argv) > 1 else Path('resources') / 'outputs' / 'books.db'

mcp_server = FastMCP('SQLite') # used to build the MCP server

@mcp_server.tool()
def list_tables() -> str:
    """List all tables in the SQLite database."""
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
        ).fetchall()
    return json.dumps([r[0] for r in rows])

@mcp_server.tool()
def describe_table(
    table_name: Annotated[
        str, Field(description='Name of the table to describe.')
    ]
) -> str:
    """Return the column definitions for a table."""
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(f'PRAGMA table_info("{table_name}")').fetchall()
    columns = [
        {'name': r[1], 'type': r[2], 'not_null': bool(r[3]), 'primary_key': bool(r[5])}
        for r in rows
    ]
    return json.dumps(columns)

# this is a general query function; 
# could provide named functions for specific controlled queries
@mcp_server.tool()
def read_query(
    query: Annotated[
        str, Field(description='A valid SQLite SELECT statement.')
    ]
) -> str:
    """Execute a SELECT SQL query and return results as JSON."""
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        cursor = conn.execute(query)
        rows = [dict(row) for row in cursor.fetchall()]
    return json.dumps(rows)

# this is a general query function; for demo only--not safe;
# could provide named functions for specific controlled queries
@mcp_server.tool() 
def write_query(
    query: Annotated[
        str,
        Field(
            description=(
                'A valid SQLite INSERT, UPDATE, or DELETE statement.'
            )
        )
    ]
) -> str:
    """Execute an INSERT, UPDATE, or DELETE statement, 
    and return number of rows affected."""
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.execute(query)
        conn.commit()
    return json.dumps({'rows_affected': cursor.rowcount})

if __name__ == '__main__':
    mcp_server.run(transport='stdio')

## Inspect the Available Tools
* `server.list_tools()` returns the schemas the agent will receive. These are generated automatically from the `@mcp_server.tool()` function docstrings and parameter metadata above.


In [ ]:
import json
import sys
from pathlib import Path
from agents.mcp import MCPServerStdio

DB_PATH = Path('resources') / 'outputs' / 'books.db'
SERVER_PATH = Path('sqlite_mcp_server.py')

if not SERVER_PATH.exists():
    raise FileNotFoundError(
        'Run the previous %%writefile sqlite_mcp_server.py cell first.')

if not DB_PATH.exists():
    raise FileNotFoundError('Run the Create the Database cell first.')

# launch server to inspect available tools
async with MCPServerStdio(
    params={
        'command': sys.executable, # launch MCP server with same enviornment as this notebook
        'args': [str(SERVER_PATH), str(DB_PATH.resolve())],
    },
    cache_tools_list=True
) as server:
    tools = await server.list_tools()

print(f'{len(tools)} tools:')
for t in tools:
    print(f'\n{t.name}: {t.description}')
    for param, schema in t.inputSchema.get('properties', {}).items():
        print(f'    param: {param} — {schema.get("description", "")}')

## Build and Run the Books Database Agent
* Agent answers natural-language questions by generating and running SQL
* No SQL code in the agent
    * Agent writes the SQL
    * MCP server handles execution

In [ ]:
import sys
from pathlib import Path
from agents import Agent, WebSearchTool, trace
from agents.mcp import MCPServerStdio
from source.agent_loop import run_conversation

DB_PATH = Path('resources') / 'outputs' / 'books.db'
SERVER_PATH = Path('sqlite_mcp_server.py')

if not SERVER_PATH.exists():
    raise FileNotFoundError('Run the Local MCP Server %%writefile cell first.')

if not DB_PATH.exists():
    raise FileNotFoundError('Run the Create the Database cell first.')

INSTRUCTIONS = """
You are a helpful assistant with two information sources:

1. A SQLite database of Deitel programming books (use the MCP database tools):
   - authors(id, first, last) — author records
   - titles(isbn, title, edition, copyright) — book records
   - author_ISBN(id, isbn) — many-to-many join linking authors to books

   Use list_tables and describe_table to explore the schema if needed.
   Use read_query for any question about Deitel books, authors, ISBNs, or editions.
   Always JOIN through author_ISBN when a question involves both authors and titles.
   Present database results in a clean, readable format, using a Markdown table 
   for multiple items.

2. Web search (use for anything the database cannot answer):
   - General programming questions and language comparisons
   - Book reviews, ratings, or recommendations from the broader community
   - Author biographies or recent news not in the database
   - Pricing, availability, or where to buy

Prefer the database for questions about Deitel books. 
Use web search for everything else.
"""

async with MCPServerStdio(
    params={
        'command': sys.executable, # launch MCP server with same enviornment as this notebook
        'args': [str(SERVER_PATH), str(DB_PATH.resolve())],
    },
    cache_tools_list=True
) as server:
    agent = Agent(
        name='BooksDatabaseAssistant',
        model='gpt-5.5',
        instructions=INSTRUCTIONS,
        tools=[WebSearchTool()],
        mcp_servers=[server]
    )
    with trace('02-05-05-local-mcp-sqlite-books-database'):
        await run_conversation(
            'Books database ready. Ask me about Deitel books — or any programming topic.',
            agent
        )

## Sample Database Prompts to Try

| SQL complexity | Prompt |
|---|---|
| Simple | `List all authors` |
| Simple | `List all books sorted by copyright year, newest first` |
| Join | `Who wrote Java How to Program?` |
| Join | `List all books by Paul Deitel` |
| Join | `Which books were co-authored by Abbey Deitel?` |
| Aggregate | `How many books did each author write? Sort by count descending` |
| Filter + join | `Show all C++ books and their authors` |
| Multi-condition | `Which books have more than two authors?` |



---
© 2026 by Deitel & Associates, Inc. All Rights Reserved.